In [64]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [65]:
ARQUIVO_RAW = "../data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

df_22_raw = pd.read_excel(ARQUIVO_RAW, sheet_name="PEDE2022")
df_23_raw = pd.read_excel(ARQUIVO_RAW, sheet_name="PEDE2023")
df_24_raw = pd.read_excel(ARQUIVO_RAW, sheet_name="PEDE2024")

# Cópias para transformação definitiva
df_22 = df_22_raw.copy()
df_23 = df_23_raw.copy()
df_24 = df_24_raw.copy()

## Funções de tratamento

In [66]:
erros_excel = [
    "#N/D",
    "#N/A",
    "#DIV/0!",
    "#VALUE!",
    "#REF!",
    "#NUM!",
    "#NAME?",
    "#NULL!",
    "INCLUIR",
    ""
]

#PADRONIZAR A COLUNA FASE PARA NUMEROS
def padronizar_fase(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().upper()

    if valor == "ALFA":
        return 0

    match = re.search(r"(\d+)", valor)

    if match:
        return int(match.group(1))

    return np.nan

#PADRONIZAR A COLUNA INSTITUICAO DE ENSINO ENTRE PUBLICO, PRIVADO E OUTROS
def padronizar_instituicao_tipo(valor):
    if pd.isna(valor):
        return "nao_informado"

    v = str(valor).strip().lower()

    if "pública" in v or "publica" in v or "escola pública" in v:
        return "publica"

    if "privada" in v or "rede decisão" in v or "jp ii" in v:
        return "privada"

    return "outros"

#IDENTIFICADO NAS BASES 2023 E 2024 SUBCATEGORIAS DENTRO DA COLUNA INSTITUICAO DE ENSINO, E PORTANTO TRATADO OS DADOS EM NOVA COLUNA

def padronizar_instituicao_subtipo(valor):
    if pd.isna(valor):
        return "nao_informado"

    v = str(valor).strip().lower()

    if "apadrinhamento" in v:
        return "apadrinhamento"

    if "parcerias com bolsa" in v or "bolsa 100" in v:
        return "parceria_bolsa"

    if "empresa parceira" in v or "pagamento por" in v:
        return "empresa_parceira"

    if "universitário" in v or "universitario" in v or "formado" in v:
        return "universitario_formado"

    if "concluiu" in v:
        return "concluiu_3em"

    if "pública" in v or "publica" in v or "escola pública" in v:
        return "sem_subtipo"

    if "privada" in v or "rede decisão" in v or "jp ii" in v:
        return "sem_subtipo"

    if "nenhuma" in v:
        return "outros"

    return "outros"


def flag_bolsa_parceria(valor):
    if pd.isna(valor):
        return 0

    v = str(valor).strip().lower()
    termos = ["apadrinhamento", "bolsa", "parceria", "parceira", "empresa"]

    return int(any(termo in v for termo in termos))


# PADRONIZAÇÃO DE GÊNERO

def padronizar_genero(valor):
    if pd.isna(valor):
        return np.nan

    v = str(valor).strip().lower()

    if v in ['menina', 'feminino']:
        return 'Feminino'

    if v in ['menino', 'masculino']:
        return 'Masculino'
        
    return 'Não informado'

In [67]:
#Valores inválidos
for df in [df_22, df_23, df_24]:
    df.replace(erros_excel, np.nan, inplace=True)

# 8.2 Ano de referência
df_22["ano_ref"] = 2022
df_23["ano_ref"] = 2023
df_24["ano_ref"] = 2024

#ID do aluno
for df in [df_22, df_23, df_24]:
    df["id_aluno"] = df["RA"].astype(str).str.extract(r"(\d+)$")

#Fase padronizada
for df in [df_22, df_23, df_24]:
    df["fase_original"] = df["Fase"]
    df["fase"] = df["Fase"].apply(padronizar_fase)
    df["flag_fase_9"] = (df["fase"] == 9).astype(int)
    df["fase"] = df["fase"].replace({9: 8})

# Instituição de ensino
for df in [df_22, df_23, df_24]:
    df["instituicao_original"] = df["Instituição de ensino"]
    df["instituicao_tipo"] = df["Instituição de ensino"].apply(padronizar_instituicao_tipo)
    df["instituicao_subtipo"] = df["Instituição de ensino"].apply(padronizar_instituicao_subtipo)
    df["flag_bolsa_parceria"] = df["Instituição de ensino"].apply(flag_bolsa_parceria)

#Genero

for df in [df_22, df_23, df_24]:
    df['genero_original'] = df['Gênero']
    df['genero'] = df['Gênero'].apply(padronizar_genero)

C:\Users\andre\AppData\Local\Temp\ipykernel_7152\1328395647.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(erros_excel, np.nan, inplace=True)


Registros classificados como Fase 9 foram identificados como casos fora da estrutura metodológica principal do PEDE, que considera fases de Alfa/0 a 8. Na camada silver, esses registros foram preservados com uma flag de rastreabilidade. Para as análises principais e modelagem preditiva, optou-se por excluí-los, evitando mistura entre alunos regulares e casos operacionais/universitários/formados.

In [68]:
# Idade padronizada

# 2022
df_22["idade_original"] = pd.to_numeric(df_22["Idade 22"], errors="coerce")
df_22["ano_nascimento"] = pd.to_numeric(df_22["Ano nasc"], errors="coerce")
df_22["idade"] = df_22["ano_ref"] - df_22["ano_nascimento"]
df_22["flag_idade_padronizada"] = (df_22["idade"] != df_22["idade_original"]).astype(int)

# 2023
df_23["idade_original"] = pd.to_numeric(df_23["Idade"], errors="coerce")
df_23["data_nasc"] = pd.to_datetime(df_23["Data de Nasc"], errors="coerce", dayfirst=True)
df_23["ano_nascimento"] = df_23["data_nasc"].dt.year
df_23["idade"] = df_23["ano_ref"] - df_23["ano_nascimento"]
df_23["flag_idade_padronizada"] = (df_23["idade"] != df_23["idade_original"]).astype(int)

# 2024
df_24["idade_original"] = pd.to_numeric(df_24["Idade"], errors="coerce")
df_24["data_nasc"] = pd.to_datetime(df_24["Data de Nasc"], errors="coerce", dayfirst=True)
df_24["ano_nascimento"] = df_24["data_nasc"].dt.year
df_24["idade"] = df_24["ano_ref"] - df_24["ano_nascimento"]
df_24["flag_idade_padronizada"] = (df_24["idade"] != df_24["idade_original"]).astype(int)

C:\Users\andre\AppData\Local\Temp\ipykernel_7152\3879267431.py:11: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_23["data_nasc"] = pd.to_datetime(df_23["Data de Nasc"], errors="coerce", dayfirst=True)


In [69]:
# Flag de ausência estrutural de avaliação
for df in [df_22, df_23, df_24]:
    df["flag_sem_avaliacao"] = (
        pd.to_numeric(df["Nº Av"], errors="coerce").fillna(0) == 0
    ).astype(int)

## Renomeação para schema comum

In [70]:
rename_22 = {
    "INDE 22": "inde",
    "Pedra 22": "pedra",
    "IAA": "iaa",
    "IEG": "ieg",
    "IPS": "ips",
    "IPP": "ipp",
    "IDA": "ida",
    "IPV": "ipv",
    "IAN": "ian",
    "Fase ideal": "fase_ideal",
    "Defas": "defasagem",
    "Ano ingresso": "ano_ingresso",
    "Matem": "matematica",
    "Portug": "portugues",
    "Inglês": "ingles"
}

rename_23 = {
    "INDE 2023": "inde",
    "INDE 23": "inde",
    "Pedra 2023": "pedra",
    "IAA": "iaa",
    "IEG": "ieg",
    "IPS": "ips",
    "IPP": "ipp",
    "IDA": "ida",
    "IPV": "ipv",
    "IAN": "ian",
    "Fase Ideal": "fase_ideal",
    "Defasagem": "defasagem",
    "Ano ingresso": "ano_ingresso",
    "Mat": "matematica",
    "Por": "portugues",
    "Ing": "ingles"
}

rename_24 = {
    "INDE 2024": "inde",
    "Pedra 2024": "pedra",
    "IAA": "iaa",
    "IEG": "ieg",
    "IPS": "ips",
    "IPP": "ipp",
    "IDA": "ida",
    "IPV": "ipv",
    "IAN": "ian",
    "Fase Ideal": "fase_ideal",
    "Defasagem": "defasagem",
    "Ano ingresso": "ano_ingresso",
    "Mat": "matematica",
    "Por": "portugues",
    "Ing": "ingles"
}

df_22 = df_22.rename(columns=rename_22)
df_23 = df_23.rename(columns=rename_23)
df_24 = df_24.rename(columns=rename_24)

In [71]:
# Garante que todas as bases tenham a coluna ipp
for df in [df_22, df_23, df_24]:
    if "ipp" not in df.columns:
        df["ipp"] = np.nan

In [72]:
for nome, df in {"2022": df_22, "2023": df_23, "2024": df_24}.items():
    df = df.loc[:, ~df.columns.duplicated()].copy()

    if nome == "2022":
        df_22 = df
    elif nome == "2023":
        df_23 = df
    else:
        df_24 = df

## Conversão de tipos

In [73]:
cols_numericas = [
    "inde", "iaa", "ieg", "ips", "ipp", "ida", "ipv", "ian",
    "idade", "idade_original", "ano_nascimento", "defasagem",
    "matematica", "portugues", "ingles"
]

for df in [df_22, df_23, df_24]:
    for col in cols_numericas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

## Decisões de remoção de colunas

Algumas colunas foram removidas da base analítica por não atenderem aos critérios de comparabilidade longitudinal, aderência às perguntas norteadoras ou relevância para modelagem.

- `Destaque IEG`, `Destaque IDA`, `Destaque IPV`: informações textuais presentes de forma inconsistente entre os anos, não utilizadas na modelagem inicial.
- `Avaliador1`, `Avaliador2`, `Rec Av1` etc.: colunas operacionais de avaliação sem continuidade suficiente entre 2022, 2023 e 2024.
- `Turma`: variável operacional de alocação, sem relação direta com as perguntas norteadoras.
- `Cg`, `Cf`, `Ct`: rankings disponíveis apenas em 2022, sem continuidade longitudinal.
- `Rec Psicologia`, `Indicado`: campos não comparáveis entre os anos.

In [74]:
colunas_silver = [
    "id_aluno",
    "ano_ref",
    "fase",
    "fase_original",
    "flag_fase_9",
    "idade",
    "idade_original",
    "flag_idade_padronizada",
    "genero_original",
    "genero",
    "ano_ingresso",
    "instituicao_original",
    "instituicao_tipo",
    "instituicao_subtipo",
    "flag_bolsa_parceria",
    "pedra",
    "inde",
    "iaa",
    "ieg",
    "ips",
    "ipp",
    "ida",
    "ipv",
    "ian",
    "fase_ideal",
    "defasagem",
    "matematica",
    "portugues",
    "ingles",
    "flag_sem_avaliacao"
]

df_22_silver = df_22.reindex(columns=colunas_silver)
df_23_silver = df_23.reindex(columns=colunas_silver)
df_24_silver = df_24.reindex(columns=colunas_silver)

df_silver = pd.concat(
    [df_22_silver, df_23_silver, df_24_silver],
    ignore_index=True
)

In [75]:
schema_resumo = pd.DataFrame({
    "coluna_2022": pd.Series(df_22_silver.columns),
    "coluna_2023": pd.Series(df_23_silver.columns),
    "coluna_2024": pd.Series(df_24_silver.columns)
})

schema_resumo

,coluna_2022,coluna_2023,coluna_2024
0,id_aluno,id_aluno,id_aluno
1,ano_ref,ano_ref,ano_ref
2,fase,fase,fase
3,fase_original,fase_original,fase_original
4,flag_fase_9,flag_fase_9,flag_fase_9
5,idade,idade,idade
6,idade_original,idade_original,idade_original
7,flag_idade_padronizada,flag_idade_padronizada,flag_idade_padronizada
8,genero_original,genero_original,genero_original
9,genero,genero,genero


In [49]:
# Validações finais da silver
# print("Linhas por ano:")
# print(df_silver["ano_ref"].value_counts().sort_index())

# print("\nFases:")
# print(df_silver["fase"].value_counts(dropna=False).sort_index())

# print("\nDuplicidades por aluno/ano:")
# duplicados = df_silver.duplicated(subset=["id_aluno", "ano_ref"]).sum()
# print(duplicados)

# print("\nPercentual de nulos principais:")
# cols_check = ["inde", "ida", "ieg", "iaa", "ips", "ipp", "ipv", "ian", "defasagem"]
# print((df_silver[cols_check].isna().mean() * 100).round(2).sort_values(ascending=False))

# print(df_22_silver.columns.equals(df_23_silver.columns))
# print(df_22_silver.columns.equals(df_24_silver.columns))

True
True


In [76]:
Path("../data/silver/").mkdir(parents=True, exist_ok=True)

df_silver.to_csv("../data/silver/base_silver.csv", index=False)